# Problem Set 3: Which Senators Does the Public Look Up?

QSS 20 · Modern Statistical Computing · Fall 2026

**Out:** Tuesday, October 13 · **Due:** Tuesday, October 20, 11:59pm

## Overview

The US Senate has 100 seats. Senators' Wikipedia articles receive
different numbers of views. A news event, a search for basic information, or
many other reasons can bring someone to an article. Wikimedia's pageviews API
reports requests for each article. These counts measure article views, not
approval of a senator or the number of distinct readers.

In this problem set **you collect the data yourself**. You will scrape the
roster of current senators from a Wikipedia page, request each senator's
July 2026 article views from the Wikimedia pageviews API, save the raw
responses, and join the two sources in DuckDB. **Which articles received the
most views in July, and how do the counts differ by party?** The roster reflects
the day you collect it; it is not a record of who served in July.

Lectures 8 and 9 cover safe requests, saved responses, HTML parsing, and
`rowspan`. Lecture 10 (Thursday, October 15) covers the DuckDB queries in
Section 4.
Work through Sections 1–3 after Lecture 9. Section 4 uses material from
Thursday's lecture; Section 5 checks the files you made along the way.

**Submission.** Copy `ps3_blank.ipynb` as `ps3_mywork.ipynb` and work in it.
Each task has an empty cell under it and a page break after it. Keep
`{{{< pagebreak >}}}` so page tagging on Gradescope stays simple. Once you're
finished, click "Kernel", "Restart Kernel and Clear Outputs of All Cells", and
then "Restart Kernel and Run All Cells", then export your notebook to PDF with
`quarto render ps3_mywork.ipynb --to typst`. Upload `ps3_mywork.ipynb` and
`ps3_mywork.pdf` to Gradescope. The Problem Set Reflections at the end are
part of the set.

Keep the raw snapshots your code collected in your working folder (see Section
5). You do not upload them. Section 5 grades the listing your notebook prints,
because it shows that someone could re-run your parse without
collecting the data again. Your code skips any file it has already saved, so
"Run All Cells" does not collect the data a second time.


**Sources.**

| Source | What it provides |
|---|---|
| [Current US senators roster](https://en.wikipedia.org/wiki/List_of_current_United_States_senators) | the roster table: state, senator, party |
| [Example API response: Tommy Tuberville, July 2026](https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/Tommy_Tuberville/monthly/20260701/20260731) | monthly article views; use the [pageviews API reference](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html) to change the request |

**Setup.** The cell below imports everything this set uses and makes the data
folders. If an import fails, run `conda activate qss20` and then
`conda install requests beautifulsoup4 python-duckdb` in a terminal.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
import requests
from bs4 import BeautifulSoup

RAW = Path('data/raw')            # what the servers sent
DERIVED = Path('data/derived')    # tables your code builds
(RAW / 'pageviews').mkdir(parents=True, exist_ok=True)
DERIVED.mkdir(parents=True, exist_ok=True)

{{< pagebreak >}}

## 1. Before any request: the collection plan (10 points)

**1.1 (4 points).** Read the [pageviews API access policy](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/documentation/access-policy.html)
and [Wikipedia's robots.txt](https://en.wikipedia.org/robots.txt). Check the
`User-agent: *` rules for the roster page's `/wiki/` path. In three or four
sentences, say whether you may request that page, what your `User-Agent` must
contain, and how you will pace the API requests. Give your planned request
count: one roster page plus one API request for each senator listed.

*Your answer:*

{{< pagebreak >}}

**1.2 (6 points).** Write your collection plan as a short list: each source,
the exact thing you will save from it, where you will save it, and the fields
you will record about the retrieval (URL, parameters, UTC time). You will be
graded in Section 5 on whether your repository matches this plan.

*Your answer:*

{{< pagebreak >}}

## 2. Scrape the roster (27 points)

Open the [current senators roster](https://en.wikipedia.org/wiki/List_of_current_United_States_senators)
and inspect the senators table. Fetch that page **once**, save the exact
bytes, and parse only your saved copy from then on.

**2.1 (5 points).** Fetch and save the page with `requests`, sending a
`User-Agent` that names the course and your own email address. Skip the
request if the file is already saved. Record the UTC retrieval time in a file.
Show the code.

{{< pagebreak >}}

**2.2 (12 points).** Parse your saved copy with BeautifulSoup. The roster
lives in `table#senators`; select it by id, not by table number. Build a
DataFrame with `state`, `senator`, `party`, and `article` (the article title
from the senator's link). Work in this order:

1. Remove footnote marks, as in lecture 9. Print the first two senator rows'
   cell counts, first-cell text, and first-cell `rowspan` attributes.
2. The state cell has `rowspan="2"` and appears only in the first row for
   each state. Save its text and remove that cell on the first row. On the
   second row, reuse the saved state without removing a cell. Then inspect
   the remaining positions for the name and party.
3. Print every distinct party label. At least one differs from `Republican`,
   `Democratic`, and `Independent`. State how you will group it for Section 4,
   then make a `party_group` column with that choice.

Hint: use `row.find_all(['th', 'td'])`; senator names are row headers (`th`).
`cells[0].get('rowspan')` returns `None` when the attribute is absent. The
party color has its own empty cell, so inspect a data row before choosing
column positions. Lecture 9 detected `rowspan` through the extra cell;
either method is acceptable. After you find the party label,
`roster['party'].replace({old_label: group_label})` can make `party_group`.

{{< pagebreak >}}

**2.3 (10 points).** Audit the parse with printed checks: the row count you
expect from civics, the number of distinct states, senators per state, party
counts from `party_group`, and duplicate or missing article titles. The Senate
has 100 seats and 50 states; a vacancy can leave fewer than 100 listed
senators. Explain why
these external counts can catch a parsing error that another read of the same
table might miss. Save the roster as `data/derived/roster.csv`.

{{< pagebreak >}}

## 3. Request the pageviews (25 points)

Open the [July 2026 API response for Tommy Tuberville](https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/Tommy_Tuberville/monthly/20260701/20260731).
The JSON contains one monthly count. Use the [pageviews API reference](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html)
to identify each part of that URL before changing the article title.

**3.1 (6 points).** Build the request for **one** senator first. Construct
the endpoint URL for monthly views of their article, `en.wikipedia`,
`all-access`, agent `user`, July 2026. The Appendix gives the URL pattern;
dates are written `YYYYMMDD`. Print the URL, then send the request, stop if
it failed (`raise_for_status()`), and display the parsed JSON. In one
sentence: why restrict the agent to `user`?

{{< pagebreak >}}

**3.2 (10 points).** Write the collection loop over the senators in your
saved roster: build each URL from its `article` field, skip a senator whose file is
already saved, pause politely between requests, save each raw response under
`data/raw/pageviews/`, and stop if any request fails (`raise_for_status()`).
After the loop, write a source record, `data/raw/source-record.json`, with
both URLs (or URL patterns), the `User-Agent`, the roster retrieval time, and
the time the pageviews collection finished (both UTC). Keep the existing
record when you rerun the notebook. Show the code and run it once.
If a request fails, keep the files already saved, fix the cause, and rerun
the loop.

Hint: repeat the single request from 3.1 in a loop over `roster['article']`.
Check `path.exists()` before requesting. After a successful response, save
`res.content` and use `time.sleep(0.5)`. Write the source record after the
loop with `json.dumps(record, indent=2)`, as in lecture 8.

{{< pagebreak >}}

**3.3 (9 points).** Parse your saved responses into a DataFrame with columns
`article` and `views`. Audit the row count, missing senators, duplicate
article titles, the `views` dtype, and the returned month. Save the table as
`data/derived/pageviews.csv`.

Hint: open one saved JSON file and inspect `payload['items'][0]` before
writing the loop. That dictionary holds `article`, `views`, and `timestamp`.
Use the returned `article` as the join key; an article title can differ from
the file name. A July 2026 monthly timestamp begins `202607`.

{{< pagebreak >}}

## 4. Join and query in DuckDB (18 points)

**4.1 (4 points).** Start an empty, in-memory DuckDB database with
`con = duckdb.connect()`. Add your two DataFrames as tables named `roster`
and `views`. No database file is needed. Use SQL to check both table counts
and the number of rows that match on `article`.

Hint: use three short `COUNT(*)` queries. For the match count, join the
tables with `USING (article)`, as in lecture 10. If fewer rows match than
the roster contains, inspect the unmatched article titles before continuing.

{{< pagebreak >}}

**4.2 (7 points).** With one SQL query (a join plus `ORDER BY ... LIMIT`),
list the ten most-viewed senators with their state, party, and July views.

{{< pagebreak >}}

**4.3 (7 points).** With one SQL query, report by party: number of senators,
mean views, and median views. Choose a party whose mean and median differ
substantially. Use one article from 4.2 to explain the gap in one or two
sentences. Only two senators are independents, so read their row with care.

{{< pagebreak >}}

## 5. Show the saved sources (10 points)

**5.1 (6 points).** Check the files your code created: `senators.html` and
`retrieved-at-utc.txt` from 2.1; the pageviews JSON files and
`source-record.json` from 3.2; and `roster.csv` and `pageviews.csv` from 2.3
and 3.3. Print whether each named file exists, the number of pageviews JSON
files, and the first five JSON file names. Keep the output visible in your
PDF. You do not upload the raw files.

{{< pagebreak >}}

**5.2 (4 points).** In two or three sentences: what can July 2026 en-Wikipedia
pageviews by `user` agents establish about the articles? Name two claims
about senators or readers that the counts cannot support.

*Your answer:*

{{< pagebreak >}}

## Problem Set Reflections (10 points)

Every problem set ends with these reflections. Answer them honestly. We grade
the specificity and honesty of your answers. You cannot use AI for this part.
We do not check spelling or grammar. Write as much as each answer needs.

Write your answer under each heading.

### AI use

List each substantive use of an AI assistant: the tool and model, what you
asked for, what it gave back, how you checked it, and whether you kept or
rejected the result. Write "None" if you used none.

*Your answer:*

### People consulted

Who you talked to about this assignment, and what about.

*Your answer:*

### Where you got stuck

The step that blocked you, and what got you unstuck.

*Your answer:*

### What you understand now

One thing you understand now that you did not when you started.

*Your answer:*

### Hours spent

An honest estimate of the time this took.

*Your answer:*

{{< pagebreak >}}

## Appendix: where the data came from

The roster comes from [Wikipedia's current US senators page](https://en.wikipedia.org/wiki/List_of_current_United_States_senators).
Monthly counts come from the [Wikimedia pageviews API](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html).
The [example request](https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/Tommy_Tuberville/monthly/20260701/20260731)
uses this pattern:
`/metrics/pageviews/per-article/en.wikipedia/all-access/user/{article}/monthly/20260701/20260731`.

Article titles come from the roster links. `requests` encodes characters
such as `é` in a URL for you.

If a senator's article title changes between your collection and grading, your
saved snapshots are the record of what you measured. That is the point of
saving them.